<a href="https://colab.research.google.com/github/Aswathi846/SAC-Based-Multi-Agent-Environmental-Mapping/blob/main/Navigation_with_Hypertuning%5BGroundtruth%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install libraries
!pip install -q gpytorch tabulate tqdm easydict
!pip install gymnasium
!pip install tensorboard
!pip install --upgrade sympy

# Create project directories
!mkdir -p Environment/Groundtruth
!mkdir -p Training
!mkdir -p Algorithm/DRL/Agent/SAC

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
  Attempting uninstall: sympy
    Found existing installation: sympy 1.13.1
    Uninstalling sympy-1.13.1:
      Successfully uninstalled sympy-1.13.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires sympy==1.13.1; python_version >= "3.9", but you have sympy 1.14.0 which is incompatible.


In [ ]:
import numpy as np
import os
import csv
from Environment.Groundtruth import AlgaeBloomGroundTruth
from Environment.Groundtruth import ShekelGroundTruth

def _load_map_from_csv(map_data_path):
    """
    Loads a map from a CSV file, attempting to auto-detect delimiter (comma or space).
    (This is your existing, robust loading function).
    """
    if map_data_path is None or not os.path.exists(map_data_path):
        raise ValueError(f"Map data path '{map_data_path}' is invalid or does not exist.")

    map_data = []
    try:
        with open(map_data_path, 'r') as f:
            reader = csv.reader(f)
            for row in reader:
                cleaned_row = [x.strip() for x in row if x.strip()]
                if not cleaned_row:
                    continue
                map_data.append([float(x) for x in cleaned_row])
        print(f"SUCCESS: Loaded {map_data_path} with ',' delimiter.")
    except (ValueError, IndexError):
        map_data = []
        try:
            with open(map_data_path, 'r') as f:
                for line in f:
                    cleaned_row = [x.strip() for x in line.split(' ') if x.strip()]
                    if not cleaned_row:
                        continue
                    map_data.append([float(x) for x in cleaned_row])
            print(f"SUCCESS: Loaded {map_data_path} with ' ' delimiter.")
        except Exception as e:
            raise ValueError(f"Failed to load map from {map_data_path}. Tried comma and space delimiters. Error: {e}")

    loaded_map = np.array(map_data, dtype=np.float32)
    return loaded_map

def get_ground_truth_map(map_type, map_size, map_data_path=None):
    """Factory function to get a ground truth map based on the type, with CSV fallback."""
    if map_type == "algae_bloom":
        print(f"Generating dynamic Algae Bloom map with size {map_size}")
        return AlgaeBloomGroundTruth.generate_algae_bloom_map(map_size)
    elif map_type == "shekel":
        print(f"Generating dynamic Shekel map with size {map_size}")
        return ShekelGroundTruth.generate_shekel_map(map_size)
    elif map_type == "static_map" and map_data_path and os.path.exists(map_data_path):
        print(f"Loading static map from CSV: {map_data_path}")
        return _load_map_from_csv(map_data_path)
    else:
        print(f"Warning: Map type '{map_type}' not found or invalid. Generating a random map.")
        return np.random.rand(*map_size)

In [ ]:
!pip install deap

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import math
import matplotlib.pyplot as plt
import os
import pandas as pd
from scipy.interpolate import NearestNDInterpolator
from collections import deque
import random
from typing import Optional

# Placeholder/Corrected GroundTruthMap classes (already provided in a previous step)
class GroundTruthMap:
    def __init__(self, map_size, map_path=None):
        self.map_size = map_size
        self.data = np.zeros(map_size)
        if map_path:
            self.load_from_file(map_path)
        if np.max(self.data) > np.min(self.data):
            self.data = (self.data - self.data.min()) / (self.data.max() - self.data.min())
        else:
            self.data = np.zeros(map_size)

    def load_from_file(self, file_path):
        try:
            self.data = pd.read_csv(file_path, header=None).values
            if self.data.shape != self.map_size:
                self.map_size = self.data.shape
            print(f"SUCCESS: Loaded static map from CSV: {file_path}")
        except FileNotFoundError:
            print(f"ERROR: Map file not found at {file_path}. Creating a dummy map.")
            self.data = np.random.rand(*self.map_size)
        except ValueError as e:
            print(f"ERROR: {e}")
            self.data = np.random.rand(*self.map_size)

class AlgaeBloomGroundTruth(GroundTruthMap):
    def __init__(self, map_size, map_seed=None):
        super().__init__(map_size)
        if map_seed is not None:
            np.random.seed(map_seed)
        center_x = np.random.uniform(0.3, 0.7) * map_size[1]
        center_y = np.random.uniform(0.3, 0.7) * map_size[0]
        radius_x = np.random.uniform(0.2, 0.4) * map_size[1]
        radius_y = np.random.uniform(0.2, 0.4) * map_size[0]
        X, Y = np.meshgrid(np.arange(map_size[1]), np.arange(map_size[0]))
        self.data = np.exp(-((X - center_x)**2 / (2 * radius_x**2) + (Y - center_y)**2 / (2 * radius_y**2)))
        if np.max(self.data) > np.min(self.data):
            self.data = (self.data - self.data.min()) / (self.data.max() - self.data.min())
        else:
            self.data = np.zeros(map_size)

class ShekelGroundTruth(GroundTruthMap):
    def __init__(self, map_size, map_seed=None):
        super().__init__(map_size)
        if map_seed is not None:
            np.random.seed(map_seed)
        self.num_peaks = 10
        self.peaks_A = np.random.rand(self.num_peaks) * 10
        self.peaks_C = np.random.rand(self.num_peaks, 2) * map_size[0]
        X, Y = np.meshgrid(np.arange(map_size[1]), np.arange(map_size[0]))
        self.data = np.zeros(map_size)
        for i in range(self.num_peaks):
            distance = np.sqrt((X - self.peaks_C[i, 0])**2 + (Y - self.peaks_C[i, 1])**2)
            self.data += self.peaks_A[i] * np.exp(-0.1 * distance**2)
        if np.max(self.data) > np.min(self.data):
            self.data = (self.data - self.data.min()) / (self.data.max() - self.data.min())
        else:
            self.data = np.zeros(map_size)


# Corrected MultiAgentMonitoring Environment Class
class MultiAgentMonitoring(gym.Env):
    metadata = {'render_modes': ['human', 'rgb_array'], 'render_fps': 4}
    def __init__(self, map_size, num_agents, movement_cost_factor, sensor_stds, max_episode_steps, map_path=None, map_class=None, map_seed=None,
                 reward_scale_factor=1.0, collision_threshold=0.05, collision_penalty=-2.0, reward_visited_cells=2.0,
                 reward_peak_discovery=10.0, penalty_time=-0.00001, penalty_revisit=0.0):
        super(MultiAgentMonitoring, self).__init__()

        self.map_size = map_size
        self.num_agents = num_agents
        self.movement_cost_factor = movement_cost_factor
        self.sensor_stds = sensor_stds
        self.reward_scale_factor = reward_scale_factor
        self.collision_threshold = collision_threshold
        self.collision_penalty = collision_penalty
        self.reward_visited_cells = reward_visited_cells
        self.reward_peak_discovery = reward_peak_discovery
        self.penalty_time = penalty_time
        self.penalty_revisit = penalty_revisit

        # --- Corrected Map Initialization Logic ---
        if map_class:
            print(f"Initializing environment with ground truth class: {map_class.__name__}...")
            self.gt_map_object = map_class(self.map_size, map_seed)
            self.gt_map = self.gt_map_object.data
            print("Map successfully generated from ground truth class.")
        elif map_path:
            print(f"Initializing environment from file path: {map_path}...")
            self.gt_map_object = GroundTruthMap(self.map_size, map_path)
            self.gt_map = self.gt_map_object.data
        else:
            raise ValueError("Must provide either a 'map_class' or a 'map_path'.")

        self.gt_map_normalized = (self.gt_map - self.gt_map.min()) / (self.gt_map.max() - self.gt_map.min())
        print("Environment successfully initialized.")

        self.peak_value = np.max(self.gt_map_normalized)
        peak_locations = np.argwhere(self.gt_map_normalized == self.peak_value)
        self.peak_location = peak_locations[0] if peak_locations.size > 0 else np.array([0, 0])

        self.agent_positions = None
        self.previous_actions = None
        self.visited_cells = np.zeros(map_size, dtype=int)
        self.sensed_data_coords = []
        self.sensed_data_values = []
        self.episode_step = 0
        self.max_episode_steps = max_episode_steps

        # Check map traversability and update collision threshold
        if np.any(self.gt_map_normalized < self.collision_threshold):
            print("Map contains non-traversable areas (obstacles).")
        else:
            print("Map is fully traversable.")
            self.collision_threshold = -1.0 # Effectively disable collision detection

        # --- CORRECTED CODE FOR SPACES ---
        # Define observation space
        # State includes fleet positions and previous actions
        obs_dim = self.num_agents * 2 + self.num_agents * 2
        # Use a single-dimensional shape for the concatenated observation
        self.observation_space = spaces.Box(low=0, high=1, shape=(obs_dim,), dtype=np.float32)

        # Define action space as a flattened vector
        # The low and high arrays MUST be reshaped to match the provided shape
        low_action = np.array([-1.0, -1.0] * self.num_agents, dtype=np.float32).reshape(self.num_agents, 2)
        high_action = np.array([1.0, 1.0] * self.num_agents, dtype=np.float32).reshape(self.num_agents, 2)
        self.action_space = spaces.Box(low=low_action, high=high_action, shape=(self.num_agents, 2), dtype=np.float32)

    def _get_observation(self):
        fleet_pos_norm = self.agent_positions / np.array([self.map_size[1], self.map_size[0]])
        prev_actions_norm = self.previous_actions if self.previous_actions is not None else np.zeros((self.num_agents, 2))
        return np.concatenate((fleet_pos_norm.flatten(), prev_actions_norm.flatten()))

    def _sense_and_record(self):
        sensed_values = []
        for agent_id in range(self.num_agents):
            x, y = self.agent_positions[agent_id, :].round().astype(int)
            x = np.clip(x, 0, self.map_size[1] - 1)
            y = np.clip(y, 0, self.map_size[0] - 1)

            true_value = self.gt_map_normalized[y, x]
            # Add Gaussian noise
            noise = np.random.normal(0, self.sensor_stds)
            sensed_value = np.clip(true_value + noise, 0, 1)

            self.sensed_data_coords.append((x, y))
            self.sensed_data_values.append(sensed_value)
            sensed_values.append((x, y, sensed_value))
        return sensed_values

    def _check_collision(self, new_positions):
        collisions = np.zeros(self.num_agents, dtype=bool)
        for i in range(self.num_agents):
            x, y = new_positions[i].round().astype(int)
            if not (0 <= y < self.map_size[0] and 0 <= x < self.map_size[1]):
                collisions[i] = True
            elif self.gt_map_normalized[y, x] < self.collision_threshold:
                collisions[i] = True
        return collisions

    def _calculate_reward(self, new_positions, old_positions, collisions):
        total_reward = 0
        peak_discovery_reward = 0
        new_cell_reward = 0
        revisit_penalty = 0

        # Calculate a reward for each agent
        for i in range(self.num_agents):
            x, y = new_positions[i].round().astype(int)
            x_old, y_old = old_positions[i].round().astype(int)

            if collisions[i]:
                total_reward += self.collision_penalty
            else:
                # Reward for visiting a new cell
                if self.visited_cells[y, x] == 0:
                    new_cell_reward += self.reward_visited_cells
                    self.visited_cells[y, x] = 1
                else:
                    revisit_penalty += self.penalty_revisit

                # Reward for being near the peak value
                distance_to_peak = np.linalg.norm(new_positions[i] - self.peak_location)
                if distance_to_peak < 10:  # Arbitrary radius for peak discovery
                    peak_discovery_reward += self.reward_peak_discovery * (1 - (distance_to_peak / 10))

        # Combine rewards
        total_reward = new_cell_reward + peak_discovery_reward + np.sum(collisions) * self.collision_penalty
        total_reward += self.penalty_time
        total_reward *= self.reward_scale_factor

        return total_reward

    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        super().reset(seed=seed)

        self.agent_positions = np.zeros((self.num_agents, 2))
        for i in range(self.num_agents):
            while True:
                initial_x = self.np_random.integers(0, self.map_size[1])
                initial_y = self.np_random.integers(0, self.map_size[0])
                if self.gt_map_normalized[initial_y, initial_x] >= self.collision_threshold:
                    self.agent_positions[i] = np.array([initial_x, initial_y])
                    break

        self.previous_actions = np.zeros((self.num_agents, 2))
        self.visited_cells = np.zeros(self.map_size, dtype=int)
        self.sensed_data_coords = []
        self.sensed_data_values = []
        self.episode_step = 0
        self.peak_discovered = False

        sensed_values = self._sense_and_record()
        self.visited_cells[self.agent_positions[:, 1].round().astype(int), self.agent_positions[:, 0].round().astype(int)] = 1

        obs = self._get_observation()
        info = {'fleet_positions': self.agent_positions, 'sensed_values': sensed_values, 'visited_cells': self.visited_cells}
        return obs, info

    def step(self, action):
        self.episode_step += 1

        old_positions = self.agent_positions.copy()
        clipped_action = np.clip(action, -1.0, 1.0)
        new_positions = self.agent_positions + clipped_action

        collisions = self._check_collision(new_positions)
        new_positions[collisions] = self.agent_positions[collisions]

        reward = self._calculate_reward(new_positions, old_positions, collisions)
        self.agent_positions = new_positions
        self.previous_actions = clipped_action

        sensed_values = self._sense_and_record()

        obs = self._get_observation()

        done = self.episode_step >= self.max_episode_steps
        truncated = False
        info = {
            'fleet_positions': self.agent_positions,
            'sensed_values': sensed_values,
            'visited_cells': self.visited_cells,
            'total_steps': self.episode_step,
            'collision_flags': collisions
        }

        return obs, reward, done, truncated, info

    def render(self, mode='human'):
        if self.render_mode is None:
            return

        fig, ax = plt.subplots(figsize=(10, 8))
        im = ax.imshow(self.gt_map_normalized, origin='lower', cmap='viridis')
        ax.set_title('Map Monitoring')
        ax.set_xlabel('X Coordinate')
        ax.set_ylabel('Y Coordinate')

        cbar = fig.colorbar(im, ax=ax)
        cbar.set_label('Normalized Value')

        ax.scatter(self.agent_positions[:, 0], self.agent_positions[:, 1], c='red', s=50, label='Agents', zorder=2)
        ax.legend()

        plt.grid(True, which='both', linestyle='--', linewidth=0.5)
        plt.show(block=False)
        plt.pause(0.1)
        plt.close(fig)

    def close(self):
        pass

In [ ]:
# --- Replay Buffer ---
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        # Ensure done is a boolean or convertible to float (0 or 1)
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        if len(self.buffer) < batch_size:
            # Raise an error instead of letting random.sample fail implicitly
            raise ValueError(f"Replay buffer has {len(self.buffer)} samples, but batch_size is {batch_size}. Not enough samples.")

        state, action, reward, next_state, done = zip(*random.sample(self.buffer, batch_size))

        return np.array(state, dtype=np.float32), \
               np.array(action, dtype=np.float32), \
               np.array(reward, dtype=np.float32), \
               np.array(next_state, dtype=np.float32), \
               np.array(done, dtype=np.float32)

    def __len__(self):
        return len(self.buffer)

In [ ]:
# --- Neural Network Architectures for SAC ---
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
import torch # Import torch here as well

class Actor(nn.Module):
    def __init__(self, obs_dim, action_dim, hidden_size=256):
        super(Actor, self).__init__()
        self.action_dim = action_dim
        self.log_std_min = -20
        self.log_std_max = 2

        self.linear1 = nn.Linear(obs_dim, hidden_size)
        self.linear2 = nn.Linear(hidden_size, hidden_size)

        self.mean_linear = nn.Linear(hidden_size, action_dim)
        self.log_std_linear = nn.Linear(hidden_size, action_dim)

    def forward(self, state):
        x = F.relu(self.linear1(state))
        x = F.relu(self.linear2(x))
        mean = self.mean_linear(x)
        log_std = self.log_std_linear(x)
        log_std = torch.clamp(log_std, self.log_std_min, self.log_std_max)
        return mean, log_std

    def sample(self, state):
        mean, log_std = self.forward(state)
        std = log_std.exp()
        normal = Normal(mean, std)
        x_t = normal.rsample() # for reparameterization trick (mean + std * N(0,1))
        y_t = torch.tanh(x_t)
        action = y_t
        log_prob = normal.log_prob(x_t)
        log_prob -= torch.log(1 - y_t.pow(2) + 1e-6)
        log_prob = log_prob.sum(1, keepdim=True)
        return action, log_prob, mean, log_std

class Critic(nn.Module):
    def __init__(self, obs_dim, action_dim, hidden_size=256):
        super(Critic, self).__init__()

        # Q1 network
        self.linear1_q1 = nn.Linear(obs_dim + action_dim, hidden_size)
        self.linear2_q1 = nn.Linear(hidden_size, hidden_size)
        self.linear3_q1 = nn.Linear(hidden_size, 1)

        # Q2 network
        self.linear1_q2 = nn.Linear(obs_dim + action_dim, hidden_size)
        self.linear2_q2 = nn.Linear(hidden_size, hidden_size)
        self.linear3_q2 = nn.Linear(hidden_size, 1)

    def forward(self, state, action):
        state_action = torch.cat([state, action], 1)

        x1 = F.relu(self.linear1_q1(state_action))
        x1 = F.relu(self.linear2_q1(x1))
        q1 = self.linear3_q1(x1)

        x2 = F.relu(self.linear1_q2(state_action))
        x2 = F.relu(self.linear2_q2(x2))
        q2 = self.linear3_q2(x2)

        return q1, q2

class Value(nn.Module):
    def __init__(self, obs_dim, hidden_size=256):
        super(Value, self).__init__()

        self.linear1 = nn.Linear(obs_dim, hidden_size)
        self.linear2 = nn.Linear(hidden_size, hidden_size)
        self.linear3 = nn.Linear(hidden_size, 1)

    def forward(self, state):
        x = F.relu(self.linear1(state))
        x = F.relu(self.linear2(x))
        value = self.linear3(x)
        return value

In [ ]:
# 2. SAC Agent Definition
import json
import optuna
import warnings
import argparse # Import argparse
import torch # Import torch for device check in argparse
import os # Import os for file operations
import numpy as np # Import numpy here
from datetime import datetime # Import datetime here
from torch.utils.tensorboard import SummaryWriter # Import SummaryWriter here
import random # Import random for set_seed
from collections import deque # Import deque for ReplayBuffer
import torch.optim as optim # Import optim here
import torch.nn.functional as F # Import F for F.mse_loss
import torch.nn as nn # Import nn for nn.utils.clip_grad_norm_

# Import network architectures from the cell where they are defined
from __main__ import Actor, Critic, Value

class SACAgent:
    def __init__(self, observation_dim, action_dim, args, writer):
        self.gamma = args.gamma
        self.tau = args.tau
        self.lr = args.lr
        # Initialize alpha as a tensor on the correct device
        self.alpha = torch.tensor(args.alpha, requires_grad=False, device=args.device)
        self.auto_entropy_tuning = args.auto_entropy_tuning
        self.target_entropy = -torch.prod(torch.Tensor([action_dim]).to(args.device)).item()
        self.hidden_size = args.hidden_size
        self.device = args.device
        self.writer = writer # TensorBoard writer

        # Correctly calculate the flattened observation dimension
        self.flattened_observation_dim = np.prod(observation_dim) # observation_dim is the shape tuple from env.observation_space.shape

        # Value Network - Corrected class name
        self.value_net = Value(self.flattened_observation_dim, self.hidden_size).to(self.device)
        self.target_value_net = Value(self.flattened_observation_dim, self.hidden_size).to(self.device)

        # Q Networks - Corrected class name
        self.q_net1 = Critic(self.flattened_observation_dim, action_dim, self.hidden_size).to(self.device)
        self.q_net2 = Critic(self.flattened_observation_dim, action_dim, self.hidden_size).to(self.device)

        self.target_q_net1 = Critic(self.flattened_observation_dim, action_dim, self.hidden_size).to(self.device)
        self.target_q_net2 = Critic(self.flattened_observation_dim, action_dim, self.hidden_size).to(self.device)

        # Policy Network - Corrected class name (assuming Actor is the policy network)
        self.policy_net = Actor(self.flattened_observation_dim, action_dim, self.hidden_size).to(self.device)

        # Copy initial weights to target networks
        for target_param, param in zip(self.target_value_net.parameters(), self.value_net.parameters()):
            target_param.data.copy_(param.data)
        for target_param, param in zip(self.target_q_net1.parameters(), self.q_net1.parameters()):
            target_param.data.copy_(target_param.data * (1.0 - self.tau) + param.data * self.tau)
        for target_param, param in zip(self.target_q_net2.parameters(), self.q_net2.parameters()):
            target_param.data.copy_(target_param.data * (1.0 - self.tau) + param.data * self.tau)


        # Optimizers
        self.value_optimizer = optim.Adam(self.value_net.parameters(), lr=self.lr)
        self.q1_optimizer = optim.Adam(self.q_net1.parameters(), lr=self.lr)
        self.q2_optimizer = optim.Adam(self.q_net2.parameters(), lr=self.lr) # Corrected this line
        self.policy_optimizer = optim.Adam(self.policy_net.parameters(), lr=self.lr)

        # Entropy tuning (alpha)
        if self.auto_entropy_tuning:
            # log_alpha should be a parameter that is optimized
            # Ensure log_alpha is initialized from the tensor alpha
            self.log_alpha = torch.tensor(np.log(self.alpha.item()), requires_grad=True, device=self.device)
            self.alpha_optimizer = optim.Adam([self.log_alpha], lr=self.lr)
        else:
             # If not auto-tuning, log_alpha is not needed, alpha is fixed as a tensor
             pass # alpha is already initialized as a tensor

        self.replay_buffer = ReplayBuffer(args.replay_size)
        self.batch_size = args.batch_size
        self.updates = 0 # Counter for optimization steps


    def select_action(self, state, evaluate=False):
        # Convert numpy state to torch tensor, add batch dimension, and flatten
        # Ensure state is float32 as expected by the network
        state = torch.FloatTensor(state).unsqueeze(0).flatten(1).to(self.device)

        if evaluate is False:
            # Sample action from the policy distribution for exploration/training
            action, log_prob, _, _ = self.policy_net.sample(state)
        else:
            # Use deterministic action (mean) for evaluation
            mean, _ = self.policy_net.forward(state) # Corrected to use forward for deterministic action
            action = torch.tanh(mean)

        # Convert back to numpy array and remove batch dimension
        return action.detach().cpu().numpy()[0]

    def update_parameters(self, memory, batch_size, updates, writer):
        # Sample a batch from replay buffer
        try:
            state_batch, action_batch, reward_batch, next_state_batch, terminated_batch = memory.sample(batch_size)
        except ValueError as e:
            print(f"Warning: Not enough samples in replay buffer for batch size {batch_size}. Skipping update.")
            return # Skip update if not enough samples

        # Convert numpy arrays to torch tensors and flatten state/next_state
        state_batch = torch.FloatTensor(state_batch).flatten(1).to(self.device) # Flatten here
        action_batch = torch.FloatTensor(action_batch).to(self.device)
        reward_batch = torch.FloatTensor(reward_batch).unsqueeze(1).to(self.device) # Ensure (batch_size, 1)
        next_state_batch = torch.FloatTensor(next_state_batch).flatten(1).to(self.device) # Flatten here
        terminated_batch = torch.FloatTensor(terminated_batch).unsqueeze(1).to(self.device) # Ensure (batch_size, 1)

        # Calculate V_target for Value Network Update
        with torch.no_grad(): # No gradients needed for target calculations
            # Sample next action from current policy for V-target calculation
            next_action, log_prob, _, _ = self.policy_net.sample(next_state_batch)

            # Extract tensor outputs from Q-networks before using torch.min
            q1_next_target, q2_next_target = self.target_q_net1(next_state_batch, next_action) # Corrected to get both outputs
            # q2_next_target, _ = self.target_q_net2(next_state_batch, next_action) # Removed redundant call

            min_q_next_target = torch.min(q1_next_target, q2_next_target) - self.alpha * log_prob # Uses self.alpha (tensor)

            # Bellman backup for the value network
            next_value = reward_batch + (1 - terminated_batch) * self.gamma * min_q_next_target

        # Value Network Update
        value_pred = self.value_net(state_batch)
        value_loss = F.mse_loss(value_pred, next_value)

        self.value_optimizer.zero_grad()
        value_loss.backward()
        nn.utils.clip_grad_norm_(self.value_net.parameters(), max_norm=1.0)
        self.value_optimizer.step()

        # Q Network Update
        # Extract tensor outputs from Q-networks before using F.mse_loss
        q1_pred, q2_pred = self.q_net1(state_batch, action_batch) # Corrected to get both outputs
        # q2_pred, _ = self.q_net2(state_batch, action_batch) # Removed redundant call

        # Use target_value_net for Q-target calculation
        with torch.no_grad():
            q_target = reward_batch + (1 - terminated_batch) * self.gamma * self.target_value_net(next_state_batch)

        q1_loss = F.mse_loss(q1_pred, q_target.detach())
        q2_loss = F.mse_loss(q2_pred, q_target.detach())

        self.q1_optimizer.zero_grad()
        q1_loss.backward()
        nn.utils.clip_grad_norm_(self.q_net1.parameters(), max_norm=1.0)
        self.q1_optimizer.step()

        self.q2_optimizer.zero_grad()
        q2_loss.backward()
        nn.utils.clip_grad_norm_(self.q_net2.parameters(), max_norm=1.0)
        self.q2_optimizer.step()


        # Policy Network Update
        # Need to re-sample actions from the current policy for the policy loss calculation
        new_action, log_prob, _, _ = self.policy_net.sample(state_batch)

        # Extract tensor outputs from Q-networks before using torch.min
        q1_new_action, q2_new_action = self.q_net1(state_batch, new_action) # Corrected to get both outputs
        # q2_new_action, _ = self.q_net2(state_batch, new_action) # Removed redundant call
        min_q_new_action = torch.min(q1_new_action, q2_new_action)

        # Policy loss: maximize (Q-value + entropy bonus)
        policy_loss = (self.alpha * log_prob - min_q_new_action).mean() # Uses self.alpha (tensor)

        self.policy_optimizer.zero_grad()
        policy_loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), max_norm=1.0)
        self.policy_optimizer.step()

        # Alpha (Entropy)
        alpha_loss = torch.tensor(0.).to(self.device) # Default alpha loss is 0
        if self.auto_entropy_tuning:
            # Alpha loss: minimize (alpha * (log_prob + target_entropy))
            alpha_loss = -(self.log_alpha * (log_prob.detach() + self.target_entropy)).mean()

            self.alpha_optimizer.zero_grad()
            alpha_loss.backward()
            self.alpha_optimizer.step()
            self.alpha = self.log_alpha.exp() # Update alpha value (tensor)

        # Soft update of target networks
        for target_param, param in zip(self.target_value_net.parameters(), self.value_net.parameters()):
            target_param.data.copy_(target_param.data * (1.0 - self.tau) + param.data * self.tau)
        for target_param, param in zip(self.target_q_net1.parameters(), self.q_net1.parameters()):
            target_param.data.copy_(target_param.data * (1.0 - self.tau) + param.data * self.tau)
        for target_param, param in zip(self.target_q_net2.parameters(), self.q_net2.parameters()):
            target_param.data.copy_(target_param.data * (1.0 - self.tau) + param.data * self.tau)

        # Log metrics to TensorBoard
        if writer is not None:
            writer.add_scalar('loss/value_loss', value_loss.item(), updates)
            writer.add_scalar('loss/q1_loss', q1_loss.item(), updates)
            writer.add_scalar('loss/q2_loss', q2_loss.item(), updates)
            writer.add_scalar('loss/policy_loss', policy_loss.item(), updates)
            # Ensure alpha is a tensor before calling .item()
            writer.add_scalar('alpha', self.alpha.item(), updates)
            writer.add_scalar('loss/alpha_loss', alpha_loss.item(), updates)
            writer.add_scalar('q_values/q1_mean', q1_pred.mean().item(), updates)
            writer.add_scalar('q_values/q2_mean', q2_pred.mean().item(), updates)
            writer.add_scalar('q_values/q_target_mean', q_target.mean().item(), updates)
            writer.add_scalar('value_net/value_mean', value_pred.mean().item(), updates)

In [ ]:
# Evaluation Function
import numpy as np
from scipy.interpolate import LinearNDInterpolator
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os
import matplotlib.pyplot as plt

# The following function should be placed in your script,
# typically before the `run_training_and_evaluation` function.

def evaluate_agent(env, agent, num_episodes, seed, log_dir, run_name):
    print("\n--- Starting Evaluation ---")

    all_rewards = []
    all_visited_counts = []

    # Store metrics for each episode
    all_episode_maes = []
    all_episode_mses = []
    all_episode_r2s = []
    all_episode_cmses = []

    for i in range(num_episodes):
        obs, info = env.reset(seed=seed + i)
        episode_reward = 0
        done = False
        truncated = False
        step = 0

        # Log positions and visited cells for visualization
        positions_log_eval_path = os.path.join(log_dir, f"{run_name}_eval_positions_ep{i}.csv")
        with open(positions_log_eval_path, 'w', newline='') as csvfile:
            positions_writer = csv.writer(csvfile)
            positions_writer.writerow(['step', 'agent_id', 'x', 'y'])
            positions = info.get("fleet_positions")
            if positions is not None:
                for j, pos in enumerate(positions):
                    positions_writer.writerow([step, j, pos[0], pos[1]])

            print(f"Evaluation episode {i} reset. Initial positions: {positions}")

            while not done and not truncated:
                # Corrected: Use agent.select_action for evaluation
                action = agent.select_action(obs, evaluate=True)
                action_reshaped_for_env = action.reshape(env.num_agents, 2)

                obs, reward, done, truncated, info = env.step(action_reshaped_for_env)
                episode_reward += reward
                step += 1

                positions = info.get("fleet_positions")
                if positions is not None:
                    for j, pos in enumerate(positions):
                        positions_writer.writerow([step, j, pos[0], pos[1]])

            # End of episode, calculate and store metrics
            visited_cells = info.get("visited_cells", np.zeros(env.map_size))
            visited_count = np.sum(visited_cells > 0)
            all_visited_counts.append(visited_count)

            # Interpolation and metric calculation from sensed data
            sensed_coords = np.array(env.sensed_data_coords)
            sensed_values = np.array(env.sensed_data_values)

            gt_map = env.gt_map_normalized

            if len(sensed_coords) > 1:
                interpolator = NearestNDInterpolator(sensed_coords, sensed_values)
                x_grid, y_grid = np.meshgrid(np.arange(env.map_size[1]), np.arange(env.map_size[0]))
                estimated_map = interpolator(x_grid, y_grid)
            else:
                estimated_map = np.full(env.map_size, np.mean(gt_map))

            mae_map = np.mean(np.abs(estimated_map - gt_map))
            mse_map = np.mean((estimated_map - gt_map)**2)
            ss_total = np.sum((gt_map - np.mean(gt_map))**2)
            ss_residual = np.sum((gt_map - estimated_map)**2)
            r2_map = 1 - (ss_residual / ss_total) if ss_total > 0 else 0

            # CMSE: Centralized Mean Squared Error
            cmse_map = np.mean((estimated_map - gt_map)**2)

            all_episode_maes.append(mae_map)
            all_episode_mses.append(mse_map)
            all_episode_r2s.append(r2_map)
            all_episode_cmses.append(cmse_map)

            # Additional metric: Variance of interpolated map
            variance_interpolated = np.var(estimated_map) if estimated_map.size > 0 else 0
            variance_gt = np.var(gt_map) if gt_map.size > 0 else 0
            print(f"Episode {i} - Variance of GT values: {variance_gt:.5f}")
            print(f"Episode {i} - Variance of Interpolated values: {variance_interpolated:.5f}")

            # Plotting logic for visualization
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))

            im1 = ax1.imshow(gt_map, cmap='viridis', origin='lower')
            ax1.set_title('True Ground Truth Map (Normalized)')
            ax1.set_xlabel('X')
            ax1.set_ylabel('Y')
            fig.colorbar(im1, ax=ax1)

            im2 = ax2.imshow(estimated_map, cmap='viridis', origin='lower')
            ax2.set_title('Estimated Map (Nearest Neighbor Interpolated)')
            ax2.set_xlabel('X')
            ax2.set_ylabel('Y')
            fig.colorbar(im2, ax=ax2)
            ax2.scatter(sensed_coords[:, 0], sensed_coords[:, 1], c='red', s=10, label='Sensed Points', zorder=2)
            ax2.legend()

            vis_path = os.path.join(log_dir, f"{run_name}_eval_map_ep{i}.png")
            fig.savefig(vis_path)
            plt.close(fig)

        all_rewards.append(episode_reward)

    print("Evaluation complete.")
    avg_reward = np.mean(all_rewards)
    avg_mae = np.mean(all_episode_maes)
    avg_mse = np.mean(all_episode_mses)
    avg_r2 = np.mean(all_episode_r2s)
    avg_cmse = np.mean(all_episode_cmses)

    # Calculate confidence intervals
    if len(all_episode_maes) > 1:
        ci_95_mae = 1.96 * np.std(all_episode_maes) / np.sqrt(len(all_episode_maes))
        ci_95_mse = 1.96 * np.std(all_episode_mses) / np.sqrt(len(all_episode_mses))
        ci_95_r2 = 1.96 * np.std(all_episode_r2s) / np.sqrt(len(all_episode_r2s))
        ci_95_cmse = 1.96 * np.std(all_episode_cmses) / np.sqrt(len(all_episode_cmses))
    else:
        ci_95_mae, ci_95_mse, ci_95_r2, ci_95_cmse = 0, 0, 0, 0

    results = {
        "Avg. Reward": avg_reward,
        "Avg. MAE (Visited Cells)": avg_mae,
        "CI 95% (MAE)": ci_95_mae,
        "MSE (Full Map - Interpolated)": avg_mse,
        "CI 95% (MSE)": ci_95_mse,
        "R^2 (Full Map - Interpolated)": avg_r2,
        "CI 95% (R^2)": ci_95_r2,
        "CMSE(X)": avg_cmse,
        "CI 95% (CMSE)": ci_95_cmse,
    }

    print("\nFinal Evaluation Results for this Run:")
    for key, value in results.items():
        if "CI 95%" in key:
            print(f"    {key}: {value:.4f}")
        else:
            print(f"{key}: {value:.4f}")

    return results

In [ ]:
!pip install -q optuna
!pip install sympy==1.13.1

  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
Using cached sympy-1.13.1-py3-none-any.whl (6.2 MB)
  Attempting uninstall: sympy
    Found existing installation: sympy 1.14.0
    Uninstalling sympy-1.14.0:
      Successfully uninstalled sympy-1.14.0


In [ ]:
import csv
import json
import optuna
import warnings
import argparse
import torch
import os
import numpy as np
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter
import random


# Helper function to set seeds for reproducibility
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
[1        torch.backends.cudnn.benchmark = False


def run_training_and_evaluation(args, map_configs):
    # Import SACAgent, MultiAgentMonitoring, and evaluate_agent from their respective cells
    from __main__ import SACAgent, MultiAgentMonitoring, evaluate_agent

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    log_base_dir_overall = os.path.join("runs_tuning", f"MultiAgentMonitoring_Tune_Overall_{timestamp}")
    os.makedirs(log_base_dir_overall, exist_ok=True)

    results_summary_overall = {}

    for map_comb_name, map_config in map_configs.items():
        print(f"\n--- Starting Optuna Hyperparameter Optimization for Map: {map_comb_name} ---")

        log_map_dir = os.path.join(log_base_dir_overall, map_comb_name)
        os.makedirs(log_map_dir, exist_ok=True)

        all_trials_results_map = []

        def objective(trial):
            trial_args = type('Args', (object,), args.__dict__.copy())
            trial_args.lr = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
            trial_args.batch_size = trial.suggest_categorical('batch_size', [64, 128, 256, 512])
            trial_args.gamma = trial.suggest_float('gamma', 0.95, 0.999)
            trial_args.reward_visited_cells = trial.suggest_float('reward_visited_cells', 0.1, 5.0)
            trial_args.reward_peak_discovery = trial.suggest_float('reward_peak_discovery', 1.0, 20.0)
            trial_args.penalty_time = trial.suggest_float('penalty_time', -0.1, -0.0001)
            trial_args.collision_penalty = trial.suggest_float('collision_penalty', -20.0, -1.0)

            current_run_seed = trial_args.seed + trial.number
            set_seed(current_run_seed)
            run_name = f"Trial{trial.number}_{datetime.now().strftime('%H-%M-%S')}"
            run_log_dir = os.path.join(log_map_dir, run_name)
            os.makedirs(run_log_dir, exist_ok=True)
            run_writer = SummaryWriter(log_dir=run_log_dir)

            positions_log_path = os.path.join(run_log_dir, "agent_positions_log.csv")
            with open(positions_log_path, 'w', newline='') as csvfile:
                positions_writer = csv.writer(csvfile)
                positions_writer.writerow(['step', 'agent_id', 'x', 'y'])

                print(f"\n--- Running Optuna Trial {trial.number} for {map_comb_name} with Seed: {current_run_seed} ---")
                print(f"Trial Hyperparameters: {trial.params}")

                # Corrected: Use map_class and map_seed instead of map_path
                env = MultiAgentMonitoring(
                    map_size=map_config["map_size"],
                    num_agents=map_config["num_agents"],
                    movement_cost_factor=trial_args.movement_cost_factor,
                    sensor_stds=map_config["sensor_stds"],
                    max_episode_steps=trial_args.max_episode_steps,
                    map_class=map_config["map_class"],
                    map_seed=map_config["map_seed"],
                    reward_scale_factor=trial_args.reward_scale_factor,
                    collision_threshold=trial_args.collision_threshold,
                    collision_penalty=trial_args.collision_penalty,
                    reward_visited_cells=trial_args.reward_visited_cells,
                    reward_peak_discovery=trial_args.reward_peak_discovery,
                    penalty_time=trial_args.penalty_time,
                    penalty_revisit=trial_args.penalty_revisit
                )

                observation_dim = env.observation_space.shape
                action_dim = np.prod(env.action_space.shape)

                print(f"Environment map size set to {env.map_size} from loaded file.")
                print(f"Environment initialized successfully.")
                print(f"Observation Dimension: {observation_dim}")
                print(f"Action Dimension: {action_dim}")

                agent = SACAgent(observation_dim, action_dim, trial_args, run_writer)
                replay_buffer = agent.replay_buffer

                print(f"TensorBoard logs for trial saved to: {run_log_dir}")
                print(f"Agent positions for trial saved to: {positions_log_path}")
                print("--- Starting Training for Trial ---")

                total_num_steps = 0
                training_episode_count = 0

                while total_num_steps < trial_args.num_steps:
                    obs, info = env.reset(seed=current_run_seed + training_episode_count)
                    episode_reward = 0
                    episode_steps = 0
                    done = False
                    truncated = False

                    while not done and not truncated:
                        if trial_args.start_steps > total_num_steps:
                            random_action = env.action_space.sample()
                            action = random_action.flatten()
                            action_reshaped_for_env = random_action
                        else:
                            action = agent.select_action(obs)
                            action_reshaped_for_env = action.reshape(map_config["num_agents"], 2)

                        next_obs, reward, done, truncated, info = env.step(action_reshaped_for_env)

                        if total_num_steps % 100 == 0:
                            positions = info.get("fleet_positions")
                            if positions is not None:
                                for i, pos in enumerate(positions):
                                    positions_writer.writerow([total_num_steps, i, pos[0], pos[1]])

                        replay_buffer.push(obs, action, reward, next_obs, done)

                        obs = next_obs
                        episode_reward += reward
                        episode_steps += 1
                        total_num_steps += 1

                        if len(replay_buffer) > trial_args.batch_size:
                            if total_num_steps >= trial_args.start_steps:
                                agent.update_parameters(replay_buffer, trial_args.batch_size, agent.updates, run_writer)
                                agent.updates += 1

                    training_episode_count += 1
                    run_writer.add_scalar('episode/reward', episode_reward, training_episode_count)
                    run_writer.add_scalar('episode/steps', episode_steps, training_episode_count)
                    print(f"Trial {trial.number}, Episode: {training_episode_count}, Steps: {total_num_steps}, Reward: {episode_reward:.4f}")

                    if training_episode_count % trial_args.eval_interval == 0 and total_num_steps >= trial_args.start_steps:
                        eval_metrics_periodic = evaluate_agent(env, agent, trial_args.num_eval_episodes, current_run_seed, run_log_dir, f"{run_name}_periodic_eval_{training_episode_count}")
                        trial.report(eval_metrics_periodic["Avg. Reward"], training_episode_count)

                        if trial.should_prune():
                            print(f"Trial {trial.number} pruned at episode {training_episode_count}.")
                            env.close()
                            run_writer.close()
                            raise optuna.exceptions.TrialPruned()

            print("--- Training Complete for Trial ---")
            eval_metrics = evaluate_agent(env, agent, trial_args.num_eval_episodes, current_run_seed, run_log_dir, run_name)
            run_writer.close()
            env.close()

            trial_results = {
                "trial_number": trial.number,
                "hyperparameters": trial.params,
                "metrics": {k: float(v) if isinstance(v, (np.float32, np.float64)) else v for k, v in eval_metrics.items()},
                "log_dir": run_log_dir
            }
            all_trials_results_map.append(trial_results)
            return eval_metrics["Avg. Reward"]

        study = optuna.create_study(direction="maximize",
                                     sampler=optuna.samplers.TPESampler(seed=args.seed),
                                     pruner=optuna.pruners.MedianPruner(
                                         n_startup_trials=args.pruner_startup_trials,
                                         n_warmup_steps=args.pruner_warmup_episodes,
                                         interval_steps=args.eval_interval
                                     ))
        study.optimize(objective, n_trials=args.num_trials, timeout=args.tuning_timeout)

        print(f"\n--- Optuna Study Complete for {map_comb_name} ---")
        print("Number of finished trials: ", len(study.trials))
        print("Best trial:")
        best_trial = study.best_trial
        print(f"   Value (Avg. Reward): {best_trial.value:.4f}")
        print("   Params: ")
        for key, value in best_trial.params.items():
            print(f"      {key}: {value}")

        best_trial_metrics = None
        for trial_result in all_trials_results_map:
            if trial_result["trial_number"] == best_trial.number:
                best_trial_metrics = trial_result["metrics"]
                break

        best_trial_info_map = {
            "map_name": map_comb_name,
            "best_value_avg_reward": best_trial.value,
            "best_hyperparameters": best_trial.params,
            "best_trial_metrics": best_trial_metrics,
            "datetime_finished": datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        }
        best_trial_path_map = os.path.join(log_map_dir, "best_trial_summary.json")
        with open(best_trial_path_map, 'w') as f:
            json.dump(best_trial_info_map, f, indent=4)
        print(f"Best trial summary for {map_comb_name} saved to {best_trial_path_map}")

        all_trials_path_map = os.path.join(log_map_dir, "all_trials_results.json")
        with open(all_trials_path_map, 'w') as f:
            json.dump(all_trials_results_map, f, indent=4)
        print(f"All trials results for {map_comb_name} saved to {all_trials_path_map}")

        results_summary_overall[map_comb_name] = {
            "Avg Reward": f"{best_trial.value:.4f}",
            "Best Hyperparameters": best_trial.params,
            "Evaluation Metrics": best_trial_metrics
        }

    print("\n--- Overall Hyperparameter Tuning Complete ---")
    print("\nSummary of Best Found Configurations per Map:")
    overall_summary_path = os.path.join(log_base_dir_overall, "overall_tuning_summary.json")
    with open(overall_summary_path, 'w') as f:
        json.dump(results_summary_overall, f, indent=4)
    print(f"Overall tuning summary saved to {overall_summary_path}")

    for map_name, metrics_and_params in results_summary_overall.items():
        print(f"\n--- {map_name} ---")
        print(f"Avg Reward: {metrics_and_params['Avg Reward']}")
        print("Best Hyperparameters:")
        for key, value in metrics_and_params['Best Hyperparameters'].items():
            print(f"   {key}: {value}")
        print("Evaluation Metrics:")
        for metric_name, value in metrics_and_params['Evaluation Metrics'].items():
            print(f"   {metric_name}: {value}")


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description="Multi-Agent Monitoring SAC Hyperparameter Tuning")
    parser.add_argument('--gamma', type=float, default=0.99, help='discount factor for reward (default: 0.99)')
    parser.add_argument('--tau', type=float, default=0.005, help='target smoothing coefficient(τ) (default: 0.005)')
    parser.add_argument('--lr', type=float, default=0.0003, help='learning rate (default: 0.0003)')
    parser.add_argument('--alpha', type=float, default=0.2, help='Temperature parameter α (default: 0.2)')
    parser.add_argument('--auto_entropy_tuning', type=bool, default=True, help='Autonomously adjust α (default: True)')
    parser.add_argument('--hidden_size', type=int, default=256, help='hidden size for neural networks (default: 256)')
    parser.add_argument('--replay_size', type=int, default=1000000, help='size of replay buffer (default: 1000000)')
    parser.add_argument('--batch_size', type=int, default=512, help='batch size for training (default: 256)')
    parser.add_argument('--num_steps', type=int, default=50000, help='maximum number of training steps PER TRIAL')
    parser.add_argument('--start_steps', type=int, default=10000, help='Steps for random action exploration (default: 1000)')
    parser.add_argument('--movement_cost_factor', type=float, default=0.005, help='Cost factor for agent movement (default: 0.005)')
    parser.add_argument('--max_episode_steps', type=int, default=1000, help='Maximum steps per episode (default: 1000)')
    parser.add_argument('--reward_scale_factor', type=float, default=1.0, help='Factor to scale the total reward by (default: 1.0)')
    parser.add_argument('--collision_threshold', type=float, default=0.05,
                        help='Normalized GT map value below which a cell is considered an obstacle (default: 0.05)')
    parser.add_argument('--collision_penalty', type=float, default=-2.0,
                        help='Reward penalty for colliding with an obstacle or map boundary (default: -2.0)')
    parser.add_argument('--reward_visited_cells', type=float, default=2.0, help='Weight for visiting new cells (default: 2.0)')
    parser.add_argument('--reward_peak_discovery', type=float, default=10.0, help='Weight for discovering high peaks (default: 10.0)')
    parser.add_argument('--penalty_time', type=float, default=0.00001, help='Penalty per time step (default: 0.00001)')
    parser.add_argument('--penalty_revisit', type=float, default=0.0, help='Penalty for revisiting cells (default: 0.0)')
    parser.add_argument('--num_eval_episodes', type=int, default=5, help='Number of episodes for evaluation after training (default: 5)')
    parser.add_argument('--seed', type=int, default=42, help='Random seed for reproducibility (default: 42)')
    parser.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu', help='Device to run on (cpu or cuda)')
    parser.add_argument('--num_trials', type=int, default=10, help='Number of different hyperparameter combinations to try (default: 10)')
    parser.add_argument('--tuning_timeout', type=int, default=None, help='Total time limit for the tuning in seconds (e.g., 3600 for 1 hour), None for no limit (default: None)')
    parser.add_argument('--eval_interval', type=int, default=100, help='Evaluate and report to Optuna every X training episodes for pruning (default: 100)')
    parser.add_argument('--pruner_startup_trials', type=int, default=5, help='Number of trials before pruning starts (default: 5)')
    parser.add_argument('--pruner_warmup_episodes', type=int, default=500, help='Number of episodes within a trial before pruning starts (default: 500)')
    args, unknown = parser.parse_known_args()

    print(f"Using device: {args.device}")

    # NEW full_map_configs dictionary using dynamic map classes
    full_map_configs = {
        "AlgaeBloom_Map": {
            "map_class": AlgaeBloomGroundTruth,
            "map_size": (50, 50),
            "map_seed": 123,
            "num_agents": 2,
            "sensor_stds": 0.05,
        },
        "Shekel_Map": {
            "map_class": ShekelGroundTruth,
            "map_size": (50, 50),
            "map_seed": 456,
            "num_agents": 2,
            "sensor_stds": 0.05,
        }
    }

    # REMOVED: The for loop that checked for and created dummy CSV map files.

    run_training_and_evaluation(args, full_map_configs)

[I 2025-08-02 16:49:59,747] A new study created in memory with name: no-name-49d6431e-bafa-459b-bcd1-9041907ef913


Using device: cpu

--- Starting Optuna Hyperparameter Optimization for Map: AlgaeBloom_Map ---

--- Running Optuna Trial 0 for AlgaeBloom_Map with Seed: 42 ---
Trial Hyperparameters: {'lr': 5.6115164153345e-05, 'batch_size': 64, 'gamma': 0.9576437314964739, 'reward_visited_cells': 0.38460969962417735, 'reward_peak_discovery': 17.457346769723767, 'penalty_time': -0.03994861032685344, 'collision_penalty': -6.546621021875136}
Initializing environment with ground truth class: AlgaeBloomGroundTruth...
Map successfully generated from ground truth class.
Environment successfully initialized.
Map contains non-traversable areas (obstacles).
Environment map size set to (50, 50) from loaded file.
Environment initialized successfully.
Observation Dimension: (8,)
Action Dimension: 4
TensorBoard logs for trial saved to: runs_tuning/MultiAgentMonitoring_Tune_Overall_2025-08-02_16-49-59/AlgaeBloom_Map/Trial0_16-49-59
Agent positions for trial saved to: runs_tuning/MultiAgentMonitoring_Tune_Overall_202

[I 2025-08-02 17:04:38,886] Trial 0 finished with value: 3069.5028412119586 and parameters: {'lr': 5.6115164153345e-05, 'batch_size': 64, 'gamma': 0.9576437314964739, 'reward_visited_cells': 0.38460969962417735, 'reward_peak_discovery': 17.457346769723767, 'penalty_time': -0.03994861032685344, 'collision_penalty': -6.546621021875136}. Best is trial 0 with value: 3069.5028412119586.


Evaluation complete.

Final Evaluation Results for this Run:
Avg. Reward: 3069.5028
Avg. MAE (Visited Cells): 0.2659
    CI 95% (MAE): 0.0328
MSE (Full Map - Interpolated): 0.1062
    CI 95% (MSE): 0.0259
R^2 (Full Map - Interpolated): -0.4708
    CI 95% (R^2): 0.3591
CMSE(X): 0.1062
    CI 95% (CMSE): 0.0259

--- Running Optuna Trial 1 for AlgaeBloom_Map with Seed: 43 ---
Trial Hyperparameters: {'lr': 1.0994335574766187e-05, 'batch_size': 64, 'gamma': 0.9589868209828182, 'reward_visited_cells': 1.590786990501735, 'reward_peak_discovery': 10.97037220101252, 'penalty_time': -0.05684869263765264, 'collision_penalty': -14.466646336237204}
Initializing environment with ground truth class: AlgaeBloomGroundTruth...
Map successfully generated from ground truth class.
Environment successfully initialized.
Map contains non-traversable areas (obstacles).
Environment map size set to (50, 50) from loaded file.
Environment initialized successfully.
Observation Dimension: (8,)
Action Dimension: 4
Te

[I 2025-08-02 17:19:03,824] Trial 1 finished with value: 539.7897782481834 and parameters: {'lr': 1.0994335574766187e-05, 'batch_size': 64, 'gamma': 0.9589868209828182, 'reward_visited_cells': 1.590786990501735, 'reward_peak_discovery': 10.97037220101252, 'penalty_time': -0.05684869263765264, 'collision_penalty': -14.466646336237204}. Best is trial 0 with value: 3069.5028412119586.


...

Evaluation complete.

Final Evaluation Results for this Run:
Avg. Reward: -3536.9922
Avg. MAE (Visited Cells): 0.2001
    CI 95% (MAE): 0.0611
MSE (Full Map - Interpolated): 0.0703
    CI 95% (MSE): 0.0243
R^2 (Full Map - Interpolated): -1.7545
    CI 95% (R^2): 0.9543
CMSE(X): 0.0703
    CI 95% (CMSE): 0.0243

--- Running Optuna Trial 8 for Shekel_Map with Seed: 50 ---
Trial Hyperparameters: {'lr': 0.0005323617594751496, 'batch_size': 64, 'gamma': 0.9659339827793105, 'reward_visited_cells': 3.675070273856514, 'reward_peak_discovery': 13.11359195574905, 'penalty_time': -0.011367447016624982, 'collision_penalty': -11.027916421922964}
Initializing environment with ground truth class: ShekelGroundTruth...
Map successfully generated from ground truth class.
Environment successfully initialized.
Map contains non-traversable areas (obstacles).
Environment map size set to (50, 50) from loaded file.
Environment initialized successfully.
Observation Dimension: (8,)
Action Dimension: 4
TensorBoa

[I 2025-08-03 00:00:05,006] Trial 8 finished with value: -8360.493892283132 and parameters: {'lr': 0.0005323617594751496, 'batch_size': 64, 'gamma': 0.9659339827793105, 'reward_visited_cells': 3.675070273856514, 'reward_peak_discovery': 13.11359195574905, 'penalty_time': -0.011367447016624982, 'collision_penalty': -11.027916421922964}. Best is trial 5 with value: -555.3821493185635.


Evaluation complete.

Final Evaluation Results for this Run:
Avg. Reward: -8360.4939
Avg. MAE (Visited Cells): 0.2465
    CI 95% (MAE): 0.1184
MSE (Full Map - Interpolated): 0.1134
    CI 95% (MSE): 0.0797
R^2 (Full Map - Interpolated): -3.4458
    CI 95% (R^2): 3.1232
CMSE(X): 0.1134
    CI 95% (CMSE): 0.0797

--- Running Optuna Trial 9 for Shekel_Map with Seed: 51 ---
Trial Hyperparameters: {'lr': 1.7345566642360933e-05, 'batch_size': 512, 'gamma': 0.9741959842218552, 'reward_visited_cells': 2.661390863971771, 'reward_peak_discovery': 9.123279348812442, 'penalty_time': -0.0974606292382649, 'collision_penalty': -17.950062887127217}
Initializing environment with ground truth class: ShekelGroundTruth...
Map successfully generated from ground truth class.
Environment successfully initialized.
Map contains non-traversable areas (obstacles).
Environment map size set to (50, 50) from loaded file.
Environment initialized successfully.
Observation Dimension: (8,)
Action Dimension: 4
TensorBoa

[I 2025-08-03 00:34:10,931] Trial 9 finished with value: -28307.728418633364 and parameters: {'lr': 1.7345566642360933e-05, 'batch_size': 512, 'gamma': 0.9741959842218552, 'reward_visited_cells': 2.661390863971771, 'reward_peak_discovery': 9.123279348812442, 'penalty_time': -0.0974606292382649, 'collision_penalty': -17.950062887127217}. Best is trial 5 with value: -555.3821493185635.


Evaluation complete.

Final Evaluation Results for this Run:
Avg. Reward: -28307.7284
Avg. MAE (Visited Cells): 0.1480
    CI 95% (MAE): 0.0599
MSE (Full Map - Interpolated): 0.0628
    CI 95% (MSE): 0.0344
R^2 (Full Map - Interpolated): -1.4619
    CI 95% (R^2): 1.3506
CMSE(X): 0.0628
    CI 95% (CMSE): 0.0344

--- Optuna Study Complete for Shekel_Map ---
Number of finished trials:  10
Best trial:
   Value (Avg. Reward): -555.3821
   Params: 
      lr: 0.0008692991511139548
      batch_size: 128
      gamma: 0.9951718375161327
      reward_visited_cells: 0.5336132600544056
      reward_peak_discovery: 4.723674385963759
      penalty_time: -0.09548179383783725
      collision_penalty: -13.818723715497978
Best trial summary for Shekel_Map saved to runs_tuning/MultiAgentMonitoring_Tune_Overall_2025-08-02_16-49-59/Shekel_Map/best_trial_summary.json
All trials results for Shekel_Map saved to runs_tuning/MultiAgentMonitoring_Tune_Overall_2025-08-02_16-49-59/Shekel_Map/all_trials_results.jso